In [ ]:
!pip install -q selenium webdriver-manager pandas

In [1]:

import pandas as pd 
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from datetime import date, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
import time, random, csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time, re, os
from datetime import datetime
from calendar import monthrange

In [5]:
def guardar_a_csv(datos, nombre_archivo="vuelos_kayak_data.csv"):
    """
    Convierte la lista de diccionarios a un DataFrame de pandas y lo guarda en un archivo CSV.
    Se fuerza el uso del punto y coma (;) como delimitador para evitar problemas de formato 
    al abrir en Excel o sistemas europeos.
    """
    if not datos:
        print("Advertencia: La lista de datos está vacía. No se creó ningún archivo CSV.")
        return

   
    df = pd.DataFrame(datos)
    
    columnas_ordenadas = [
        'fecha_vuelo',
        'precio', 
        'aerolinea', 
        'hora_salida', 
        'hora_llegada', 
        'origen_iata', 
        'destino_iata', 
        'paradas'
    ]
    

    columnas_a_usar = [col for col in columnas_ordenadas if col in df.columns]
    df = df[columnas_a_usar]

    df.to_csv(
        nombre_archivo,
        mode='a',            
        header=not os.path.exists(nombre_archivo),  
        index=False,
        encoding='utf-8',
        sep=';'
)
    
    

In [6]:
cities = ["MAD","LON","PAR","BER","ROM","BUH","VIE","AMS","CPH","STO","OSL","HEL","LIS","IST","ZRH","BRU","DUB","PRG","BUD","ATH", "MIL"]

# 2) pares
#pairs = [(o,d) for o in cities for d in cities if o != d]
pairs = [(cities[i], cities[j]) for i in range(len(cities)) for j in range(i+1, len(cities))]
# 3) fechas salida (cada 6 dias)
start = date(2025,12,1)
end = date(2026,8,31)
delta = (end - start).days
depart_dates = [start + timedelta(days=i) for i in range(0, delta+1, 6)]

random.seed(42)  
start_year, start_month = 2025, 12
end_year, end_month = 2026, 8
k_per_month = 3   # <-- 3 fechas aleatorias por mes

# genera lista de meses entre diciembre 2025 y agosto 2026
months = []
y, m = start_year, start_month
while (y < end_year) or (y == end_year and m <= end_month):
    months.append((y, m))
    if m == 12:
        y += 1
        m = 1
    else:
        m += 1

# -----------------------------------
# GENERAR FECHAS POR MES
# -----------------------------------

def fechas_por_mes(year, month, k):
    """Devuelve K fechas aleatorias en (year, month)."""
    days_in_month = monthrange(year, month)[1]
    all_dates = [date(year, month, d) for d in range(1, days_in_month + 1)]
    if len(all_dates) <= k:
        return all_dates
    return sorted(random.sample(all_dates, k))

# ---------------------------------------------
# GENERAR 3 FECHAS POR MES PARA CADA PAR
# ---------------------------------------------

fechas_por_par = {}

for origin, dest in pairs:
    lista_fechas = []
    for (y,m) in months:
        fechas_m = fechas_por_mes(y, m, k_per_month)
        lista_fechas.extend(fechas_m)
    fechas_por_par[(origin, dest)] = sorted(lista_fechas)

print(fechas_por_par[(pairs[2][0], pairs[2][1])][4])

def build_kayak_url(o, d, dep):
    dep_s = dep.strftime("%Y-%m-%d")
    return f"https://www.kayak.es/flights/{o}-{d}/{dep_s}?sort=bestflight_a"

url = build_kayak_url(pairs[2][0], pairs[2][1], depart_dates[6])
print(url)

2026-01-09
https://www.kayak.es/flights/MAD-BER/2026-01-06?sort=bestflight_a


In [ ]:


# 1) Opciones de Chrome - NO usar headless en local para evitar detección
opts = Options()  

# Anti-detección útiles
opts.add_argument("--disable-blink-features=AutomationControlled")
opts.add_experimental_option("excludeSwitches", ["enable-automation"])
opts.add_experimental_option("useAutomationExtension", False)
opts.add_argument("--window-size=1366,768")
opts.add_argument("--start-maximized")
opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36")

# 2) Iniciar driver con webdriver-manager (se encargará de bajar chromedriver compatible)
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=opts)

# Opcional: ocultar navigator.webdriver
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": """
        Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
        window.navigator.chrome = { runtime: {} };
        Object.defineProperty(navigator, 'languages', {get: () => ['en-US','en']});
        Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3]});
    """
})

def click_Aceptar_todo(timeout=10):
    # 1) Intentar encontrar por XPATH (texto exacto normalizado)
    xpath_text = "//*[normalize-space(text())='Aceptar todo']"
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, xpath_text))
        )
        clase_del_elemento = el.get_attribute('class')
        print(f"✅ Encontrado: Elemento con texto 'Aceptar todo'.")
        print(f"   Clase CSS del elemento: **{clase_del_elemento}**")
        # Si el div está dentro de un <button>, clicar el ancestro <button>
        try:
            btn = el.find_element(By.XPATH, "ancestor::button[1]")
            btn.click()
        except Exception:
            # si no hay button, clicar el propio div
            el.click()
        print("Clicked 'Aceptar todo' (direct match).")
        return True
    except TimeoutException:
        pass

    # 2) Intentar por clase (fallback)
    try:
        el2 = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.RxNS-button-content"))
        )
        # asegurar que el texto sea el que queremos
        if el2.text.strip().lower() == "aceptar todo":
            try:
                btn = el2.find_element(By.XPATH, "ancestor::button[1]")
                btn.click()
            except Exception:
                el2.click()
            print("Clicked 'Aceptar todo' (by class).")
            return True
    except TimeoutException:
        pass

    # 3) Si no aparece en el DOM principal, buscar en iframes
    frames = driver.find_elements(By.TAG_NAME, "iframe")
    for f in frames:
        try:
            driver.switch_to.frame(f)
            try:
                elf = driver.find_element(By.XPATH, xpath_text)
                try:
                    btn = elf.find_element(By.XPATH, "ancestor::button[1]")
                    btn.click()
                except Exception:
                    elf.click()
                driver.switch_to.default_content()
                print("Clicked 'Aceptar todo' inside iframe.")
                return True
            except Exception:
                driver.switch_to.default_content()
                continue
        except Exception:
            driver.switch_to.default_content()
            continue

    print("No se encontró el botón 'Aceptar todo'.")
    return False


def esperar_carga_vuelos(driver, contenedor_clase, timeout=40):
    """
    Espera hasta que el indicador de carga desaparezca Y al menos un resultado de vuelo aparezca.
    """
    print("Esperando la carga completa de vuelos...")
    
    # 1. Esperar a que el spinner desaparezca (clase común en Kayak es 'NnB_N')
    try:
        WebDriverWait(driver, timeout).until_not(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'div.NnB_N'))
        )
        print("Spinner de carga desaparecido.")
    except TimeoutException:
        print("Advertencia: Timeout esperando que desaparezca el spinner.")

    # 2. Esperar a que el primer resultado de vuelo aparezca
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"div.{contenedor_clase}"))
        )
        return True
    except TimeoutException:
        print(f"❌ Error: No se encontraron resultados de vuelos con la clase '{contenedor_clase}' después de la espera.")
        return False


def extraer_contenedores(driver, contenedor_clase):
    """
    Localiza todos los elementos contenedores de vuelo y devuelve la lista.
    """
    try:
        # Encontrar todos los contenedores de vuelo usando tu clase
        contenedores = driver.find_elements(By.CSS_SELECTOR, f"div.{contenedor_clase}")
        return contenedores
        
    except Exception as e:
        print(f"Error al buscar contenedores: {e}")
        return []
        

def extraer_datos_vuelo_completo(contenedor, fecha_vuelo_global): 
    """
    Extrae todos los datos de un contenedor de vuelo, usando la clase de precio 
    como filtro principal para descartar anuncios y asignando la fecha global.
    """
    # Clases identificadas para extracción
    CLASE_PRECIO = "e2GB-price-text"
    CLASE_HORARIOS_CONT = "vmXl.vmXl-mod-variant-large"
    CLASE_AEROLINEA = "J0g6-operator-text"
    CLASE_CIUDAD_SALIDA = "jLhY-airport-info"
    CLASE_DIRECTO_CONT = "vmXl.vmXl-mod-variant-default"

    # Diccionario para almacenar los resultados (Valores por defecto)
    vuelo = {
        'tipo': 'VUELO',
        'fecha_vuelo': fecha_vuelo_global, # <-- 2. ASIGNACIÓN INICIAL DE LA FECHA GLOBAL
        'precio': None,
        'hora_salida': 'N/A',
        'hora_llegada': 'N/A',
        'aerolinea': 'N/A',
        'origen_iata': 'N/A',
        'destino_iata': 'N/A',
        'paradas': 'N/A'
    }

    try:
        # ----------------------------------------------------
        # 1. FILTRADO OBLIGATORIO: Comprobar el precio. Si falla, va al except principal.
        # ----------------------------------------------------
        precio_el = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_PRECIO}")
        
        # --- PRECIO (Limpieza) ---
        precio_bruto = precio_el.text.strip()
        precio_limpio = re.sub(r'[^\d,]', '', precio_bruto).replace(',', '.')
        try:
            vuelo['precio'] = float(precio_limpio)
        except ValueError:
            vuelo['precio'] = precio_bruto 
            
        # ----------------------------------------------------
        # 2. EXTRACCIÓN DE LOS DEMÁS CAMPOS (Usando try/except anidado para robustez)
        # ----------------------------------------------------

        # --- HORARIOS (Salida y Llegada) ---
        try:
            horarios_cont = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_HORARIOS_CONT.replace(' ', '.')}")
            tiempos_span = horarios_cont.find_elements(By.TAG_NAME, "span")
            if len(tiempos_span) >= 1:
                vuelo['hora_salida'] = tiempos_span[0].text.strip()
            if len(tiempos_span) >= 3:
                vuelo['hora_llegada'] = tiempos_span[2].text.strip()
        except NoSuchElementException:
            pass # Si falla, usa los valores por defecto ('N/A')

        # --- AEROLÍNEA ---
        try: 
            aerolinea_el = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_AEROLINEA}")
            vuelo['aerolinea'] = aerolinea_el.text.strip()
        except NoSuchElementException:
            pass

        # --- ORIGEN y DESTINO (Código IATA) ---
        try: 
            # Se buscan ambos códigos IATA de una vez
            destino_info_spans = contenedor.find_elements(By.CSS_SELECTOR, f"span.{CLASE_CIUDAD_SALIDA}")
            
            # Origen (primer span)
            if len(destino_info_spans) >= 1:
                origen_iata_el = destino_info_spans[0].find_element(By.TAG_NAME, "span")
                vuelo['origen_iata'] = origen_iata_el.text.strip()

            # Destino (segundo span)
            if len(destino_info_spans) >= 2:
                 destino_iata_el = destino_info_spans[1].find_element(By.TAG_NAME, "span")
                 vuelo['destino_iata'] = destino_iata_el.text.strip()
        except NoSuchElementException:
            pass
            
        # --- DIRECTO O ESCALAS ---
        try: 
            escalas_el = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_DIRECTO_CONT.replace(' ', '.')}")
            vuelo['paradas'] = escalas_el.text.strip()
        except NoSuchElementException:
            pass
            
        return vuelo
        
    except NoSuchElementException:
        # Esto solo se ejecuta si falla la comprobación inicial del precio (FILTRADO DE ANUNCIOS)
        return {'tipo': 'ANUNCIO/PROMO'}
    
    except Exception as e:
        print(f"Error inesperado al procesar un contenedor: {e}")
        return {'tipo': 'ERROR'}


CLASE_CONTENEDOR_FECHA = "NITa-date-inline"
CLASE_VALOR_FECHA = "NITa-value"
def extraer_fecha_global(driver, clase_contenedor, clase_valor, timeout=5):
    """
    Busca la fecha de la búsqueda usando una estrategia de búsqueda anidada:
    1. Busca el contenedor principal (Ej: div.NITa-date-inline).
    2. Busca el span del valor (Ej: span.NITa-value) dentro de ese contenedor.
    """
    try:
        # 1. Esperar a que el contenedor principal esté presente
        contenedor_el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"div.{clase_contenedor}"))
        )
        
        
        # 2. Buscar el elemento de valor dentro del contenedor
        # Usamos find_element normal, y si falla, el bloque except principal lo capturará.
        fecha_el = contenedor_el.find_element(By.CSS_SELECTOR, f"span.{clase_valor}")
        
        fecha_texto = fecha_el.text.strip()
        return fecha_texto
        
    except TimeoutException:
        print(f"❌ Error: El contenedor de fecha 'div.{clase_contenedor}' no apareció después de {timeout} segundos.")
        return 'N/A'
    except NoSuchElementException:
        # Esto ocurre si el contenedor se encontró, pero el span.NITa-value no estaba dentro (improbable en este caso, pero seguro).
        print(f"❌ Error: El elemento de valor 'span.{clase_valor}' no se encontró dentro del contenedor.")
        return 'N/A'

start_origin = "PAR"
start_dest = "LIS"  
start_date = date(2026, 2, 20)
skip = True

for origin, dest in pairs:                     # 1. Loop por ciudades
    for dep in fechas_por_par[(origin, dest)]:                  # 2. Loop por fechas de salida

        # Saltar hasta llegar a donde nos hemos quedado
        if skip:
            if origin == start_origin and dest == start_dest and dep >= start_date:
                skip = False   # <-- aquí empieza el scraping real
                print("\n🔵 Reanudando aquí:", origin, dest, dep)
            else:
                continue

        
        url = build_kayak_url(origin, dest, dep)   # 3. URL para este vuelo
        print("Scrapeando:", url)

        try:
            driver.get(url)
            click_Aceptar_todo()          
            time.sleep(1.5)

            # 2. EXTRAER LA FECHA GLOBAL UNA SOLA VEZ      
            fecha_de_busqueda = extraer_fecha_global(driver, CLASE_CONTENEDOR_FECHA, CLASE_VALOR_FECHA)

            # 3. Esperar a que aparezcan los resultados de vuelo
            CLASE_CONTENEDOR_VUELO = "Fxw9-result-item-container"
            vuelos_finales = []
            elementos_vuelo = []   # <-- importante: inicializar para evitar NameError

            if esperar_carga_vuelos(driver, CLASE_CONTENEDOR_VUELO):
                # 4. Obtener la lista de contenedores
                elementos_vuelo = extraer_contenedores(driver, CLASE_CONTENEDOR_VUELO)

                # 5. ITERACIÓN Y EXTRACCIÓN CON FILTRADO
                print("\n--- INICIANDO EXTRACCIÓN Y FILTRADO DE ANUNCIOS ---")
                for i, contenedor in enumerate(elementos_vuelo):
                    # Pasar la fecha extraída globalmente
                    datos = extraer_datos_vuelo_completo(contenedor, fecha_de_busqueda)

                    if datos.get('tipo') == 'VUELO':
                        datos.pop('tipo', None)
                        datos.setdefault('origen_iata', origin)
                        datos.setdefault('destino_iata', dest)
                        datos.setdefault('fecha_vuelo', dep.isoformat())
                        vuelos_finales.append(datos)

            else:
                print("⚠️ No se cargaron vuelos (esperar_carga_vuelos devolvió False).")

            print(f"\nExtracción finalizada. Total de vuelos guardados (filtrados): {len(vuelos_finales)}")

            # 6. GUARDAR A CSV 
            if vuelos_finales:
                guardar_a_csv(vuelos_finales)
            else:
                print("No hay vuelos para guardar en este intento.")

        except Exception as e:
            # imprime el origen/dest/dep para facilitar debugging
            print("ERROR →", origin, dest, dep, e)

        # 7. Rate limit
        time.sleep(random.uniform(2, 5))


driver.quit()


🔵 Reanudando aquí: PAR LIS 2026-02-20
Scrapeando: https://www.kayak.es/flights/PAR-LIS/2026-02-20?sort=bestflight_a
✅ Encontrado: Elemento con texto 'Aceptar todo'.
   Clase CSS del elemento: **RxNS-button-content**
Clicked 'Aceptar todo' (direct match).
Esperando la carga completa de vuelos...
Spinner de carga desaparecido.

--- INICIANDO EXTRACCIÓN Y FILTRADO DE ANUNCIOS ---

Extracción finalizada. Total de vuelos guardados (filtrados): 15
Scrapeando: https://www.kayak.es/flights/PAR-LIS/2026-02-24?sort=bestflight_a
No se encontró el botón 'Aceptar todo'.
❌ Error: El contenedor de fecha 'div.NITa-date-inline' no apareció después de 5 segundos.
Esperando la carga completa de vuelos...
Spinner de carga desaparecido.
❌ Error: No se encontraron resultados de vuelos con la clase 'Fxw9-result-item-container' después de la espera.
⚠️ No se cargaron vuelos (esperar_carga_vuelos devolvió False).

Extracción finalizada. Total de vuelos guardados (filtrados): 0
No hay vuelos para guardar en es

In [ ]:
driver.quit()